## Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairing of:

1. A schema, including the name of the tool, a description, and/or argument definitions
2. A function or coroutine to execute

In [2]:
import os
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model = "nvidia/nemotron-3-super-120b-a12b:free",
    api_key = os.environ["OPENROUTER_API_KEY"],
    base_url = "https://openrouter.ai/api/v1"
)

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str)-> str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])


In [4]:
response = model_with_tools.invoke("Whats the weather like in Boston?")
print(response)
print(response.content)
for tool_call in response.tool_calls:
    print(f"Tool : {tool_call['name']}")
    print(f"Tool : {tool_call['args']}")


content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 275, 'total_tokens': 343, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 48, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-super-120b-a12b-20230311:free', 'system_fingerprint': None, 'id': 'gen-1782471329-f6YQjSVcHCAGNPdfPSkB', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019f0392-0722-7943-b9e2-21d420fe08b5-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call-230b048d-e190-4e02-b41a-409f7f49d747', 'type': 'tool_call'}] invalid_to

### Tool Execution Loop

In [5]:
# Step-1 : Model generates tool calls
messages = [{"role" : "user", "content" : "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step-2 : Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)
#Step-3 : Pass the result back to the model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

It's sunny in Boston.


In [6]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 275, 'total_tokens': 331, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 34, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-super-120b-a12b-20230311:free', 'system_fingerprint': None, 'id': 'gen-1782471337-7PVdnlRmUlR0MAuOdb4C', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f0392-2561-7e51-860c-2f9f15fc3063-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id':